# LeetCode 2212 — Maximum Points in an Archery Competition bằng Python

Link đề: https://leetcode.com/problems/maximum-points-in-an-archery-competition/ · Độ khó: **Medium** · Chủ đề: Array, Backtracking, Bit Manipulation, Enumeration, Dynamic Programming (Knapsack)

Skeleton LeetCode:

```python
class Solution:
    def maximumBobPoints(self, numArrows: int, aliceArrows: List[int]) -> List[int]:
        ...
```

## 1. Mục tiêu học

1. Nhận ra cấu trúc **"được/mất từng ô"**: để thắng ô `k`, Bob cần **đúng** `aliceArrows[k] + 1` mũi tên (bắn nhiều hơn là lãng phí), không thắng thì bắn `0`.
2. Quy bài về **cái túi 0/1** với 12 món đồ: món `k` có "trọng lượng" `a[k] + 1` và "giá trị" `k`, sức chứa `numArrows`.
3. Giải bằng **liệt kê bitmask** `2¹²` tập con (cực nhanh vì chỉ 12 ô), bằng **quay lui**, và bằng **DP cái túi** có truy vết.
4. Xử lý chi tiết đầu ra: tổng mũi tên của Bob phải **đúng bằng** `numArrows` — dồn mũi tên thừa vào một ô bất kỳ.
5. Kiểm thử bài **nhiều đáp án** bằng hàm kiểm tra hợp lệ + so điểm với vét cạn mọi cách chia mũi tên.

## 2. Đề bài & ví dụ

Alice và Bob thi bắn cung. Bia có 12 vùng điểm `0..11`. Alice bắn `numArrows` mũi trước (cho bởi `aliceArrows[k]` = số mũi vào vùng `k`), sau đó Bob bắn `numArrows` mũi.

Với mỗi vùng `k`: nếu `a_k ≥ b_k` thì **Alice** được `k` điểm; nếu `a_k < b_k` thì **Bob** được `k` điểm; nếu `a_k = b_k = 0` thì không ai được.

Bob muốn **tối đa** điểm của mình. Trả về mảng `bobArrows` (12 phần tử, **tổng đúng bằng `numArrows`**). Nhiều đáp án thì trả về bất kỳ.

**Ví dụ 1:** `numArrows = 9`, `aliceArrows = [1,1,0,1,0,0,2,1,0,1,2,0]` → ví dụ `[0,0,0,0,1,1,0,0,1,2,3,1]`, Bob được `4+5+8+9+10+11 = 47`.

**Ví dụ 2:** `numArrows = 3`, `aliceArrows = [0,0,1,0,0,0,0,0,0,0,0,2]` → ví dụ `[0,0,0,0,0,0,0,0,1,1,1,0]`, Bob được `8+9+10 = 27`.

**Ràng buộc:** `1 ≤ numArrows ≤ 10⁵`, `sum(aliceArrows) == numArrows`.

In [1]:
import random
import time
from functools import lru_cache
from itertools import combinations
from typing import List

examples = [
    (9, [1, 1, 0, 1, 0, 0, 2, 1, 0, 1, 2, 0], 47),
    (3, [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 2], 27),
]

def bob_score(alice, bob):
    return sum(k for k in range(12) if bob[k] > alice[k])

def is_valid(numArrows, alice, bob, best):
    """Hợp lệ: 12 số không âm, tổng đúng numArrows, đạt điểm tối ưu best."""
    return len(bob) == 12 and all(x >= 0 for x in bob) and sum(bob) == numArrows and bob_score(alice, bob) == best

for n, a, best in examples:
    print(f"numArrows={n}, alice={a} -> điểm tối ưu {best}")

numArrows=9, alice=[1, 1, 0, 1, 0, 0, 2, 1, 0, 1, 2, 0] -> điểm tối ưu 47
numArrows=3, alice=[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 2] -> điểm tối ưu 27


## 3. Kiến thức nền

### 3.1 Điều kiện thắng một vùng và "giá" của nó

Bob thắng vùng `k` ⇔ `b_k > a_k` ⇔ `b_k ≥ a_k + 1`. Bắn thêm vào vùng đã thắng không cho thêm điểm ⇒ **giá** để thắng vùng `k` là đúng `a_k + 1` mũi. Hòa (`b_k = a_k`) thuộc về Alice.

In [2]:
alice = examples[0][1]
cost = [a + 1 for a in alice]
print("giá để thắng từng vùng k:", list(enumerate(cost)))

giá để thắng từng vùng k: [(0, 2), (1, 2), (2, 1), (3, 2), (4, 1), (5, 1), (6, 3), (7, 2), (8, 1), (9, 2), (10, 3), (11, 1)]


### 3.2 Bitmask duyệt tập con 12 phần tử

Mỗi số `mask ∈ [0, 2¹²)` biểu diễn một tập vùng Bob muốn thắng: bit `k` bật ⇔ thắng vùng `k`. `4096` tập — duyệt hết trong tích tắc.

In [3]:
mask = 0b111100110000
print([k for k in range(12) if mask >> k & 1], "| tổng số tập:", 1 << 12)

[4, 5, 8, 9, 10, 11] | tổng số tập: 4096


### 3.3 Bài toán cái túi 0/1

Có các món đồ (trọng lượng `w_k`, giá trị `v_k`) và sức chứa `W`. Chọn tập con có tổng trọng lượng `≤ W` để tổng giá trị lớn nhất. DP `dp[c]` = giá trị tốt nhất với sức chứa `c`, duyệt `c` **giảm dần** cho mỗi món (để mỗi món dùng tối đa một lần).

In [4]:
def knapsack_value(weights, values, W):
    dp = [0] * (W + 1)
    for w, v in zip(weights, values):
        for c in range(W, w - 1, -1):
            dp[c] = max(dp[c], dp[c - w] + v)
    return dp[W]

print(knapsack_value([2, 3, 4], [3, 4, 6], 5), "(chọn món nặng 2 và 3)")

7 (chọn món nặng 2 và 3)


### 3.4 Mũi tên thừa

Nếu tập vùng đã chọn tốn ít hơn `numArrows`, phần thừa phải được bắn ở đâu đó (tổng phải đúng `numArrows`). Dồn hết vào vùng `0` là an toàn: vùng 0 đáng 0 điểm nên không làm thay đổi điểm (và dồn vào một vùng đã thắng cũng không sao).

In [5]:
bob = [0] * 12; bob[11] = 1; numArrows = 4
bob[0] += numArrows - sum(bob)
print(bob, "tổng =", sum(bob))

[3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1] tổng = 4


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 2: `numArrows = 3`, Alice bắn `[0,0,1,0,...,0,2]` (1 mũi vào vùng 2, 2 mũi vào vùng 11).

| Vùng k | a_k | Giá thắng (a_k+1) | Điểm |
|---|---|---|---|
| 11 | 2 | 3 | 11 |
| 10 | 0 | 1 | 10 |
| 9 | 0 | 1 | 9 |
| 8 | 0 | 1 | 8 |
| 2 | 1 | 2 | 2 |
| các vùng khác | 0 | 1 | k |

Với 3 mũi: thắng vùng 11 (tốn 3) được 11 điểm; hoặc thắng 10, 9, 8 (mỗi vùng tốn 1) được 27 điểm ⇒ chọn cái sau. Đây đúng là bài **cái túi**: chọn tập vùng có tổng giá `≤ 3` và tổng điểm lớn nhất.

Tham lam theo "điểm lớn nhất trước" (lấy 11) cho 11 điểm — **sai**; tham lam theo "tỉ lệ điểm/giá" có thể đúng ở đây nhưng không đúng nói chung (mục 13). Vì chỉ có 12 vùng, cứ thử **hết** `2¹²` tập.

In [6]:
n, a, _ = examples[1]
for S in [(11,), (10, 9, 8), (11, 10), (10, 9, 8, 7)]:
    c = sum(a[k] + 1 for k in S)
    print(f"tập {S}: giá {c} {'<=' if c <= n else '>'} {n} -> điểm {sum(S) if c <= n else 'không đủ tên'}")

tập (11,): giá 3 <= 3 -> điểm 11
tập (10, 9, 8): giá 3 <= 3 -> điểm 27
tập (11, 10): giá 4 > 3 -> điểm không đủ tên
tập (10, 9, 8, 7): giá 4 > 3 -> điểm không đủ tên


## 5. Cách 1 — Brute Force / baseline: vét cạn mọi cách chia mũi tên

**Ý tưởng:** liệt kê mọi cách chia `numArrows` mũi vào 12 vùng (bài toán "chia kẹo Euler": `C(numArrows + 11, 11)` cách), tính điểm, lấy max. Không dùng bất kỳ nhận xét nào — chuẩn tuyệt đối, chỉ chạy được với `numArrows` rất nhỏ.

In [7]:
def all_distributions(total, parts):
    if parts == 1:
        yield (total,)
        return
    for x in range(total + 1):
        for rest in all_distributions(total - x, parts - 1):
            yield (x,) + rest

def bob_brute(numArrows, alice):
    best, best_bob = -1, None
    for bob in all_distributions(numArrows, 12):
        s = bob_score(alice, bob)
        if s > best:
            best, best_bob = s, list(bob)
    return best_bob

for n, a, best in examples[1:]:
    got = bob_brute(n, a); print(got, "điểm", bob_score(a, got)); assert is_valid(n, a, got, best)

[0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0] điểm 27


**Độ phức tạp:** `C(N + 11, 11)` cách chia — với `N = 3` là `364`, `N = 9` đã là `167960`, `N = 20` là `≈ 8,6·10⁷`. Không thể với `N = 10⁵`.

**Nút thắt:** phần lớn các cách chia là "lãng phí" (bắn nhiều hơn mức cần ở một vùng, hoặc bắn ít hơn mức cần — thua vẫn tốn tên). Theo 3.1, ở mỗi vùng chỉ có **hai** lựa chọn đáng xét: `0` hoặc `a_k + 1`.

In [8]:
from math import comb
for N in (3, 9, 20, 100):
    print(f"numArrows={N}: số cách chia = C(N+11, 11) = {comb(N + 11, 11)}")

numArrows=3: số cách chia = C(N+11, 11) = 364
numArrows=9: số cách chia = C(N+11, 11) = 167960
numArrows=20: số cách chia = C(N+11, 11) = 84672315
numArrows=100: số cách chia = C(N+11, 11) = 473239787751081


## 6. Ý tưởng thuật toán / bước đột phá

**Bổ đề (chuẩn hóa).** Từ một phương án tối ưu bất kỳ, có thể biến đổi thành phương án mà mỗi vùng `k` có `b_k ∈ {0, a_k + 1}`, trừ vùng `0` nhận phần thừa, mà **không giảm điểm**:
- vùng Bob thắng với `b_k > a_k + 1`: giảm về `a_k + 1` (vẫn thắng), chuyển phần dư sang vùng 0;
- vùng Bob thua (`b_k ≤ a_k`): giảm về 0 (vẫn thua), chuyển phần dư sang vùng 0.

Vùng 0 trị giá 0 điểm nên phần dư không ảnh hưởng.

⇒ Bài toán = chọn tập `S ⊆ {1..11}` (vùng 0 vô nghĩa) với `Σ_{k∈S} (a_k + 1) ≤ numArrows`, tối đa `Σ_{k∈S} k`. Đây là **cái túi 0/1** 11 món.

**Quy tắc "nếu… thì…" khi duyệt bitmask:**
- **Nếu** tổng giá của tập vượt `numArrows` **thì** bỏ qua.
- **Nếu** điểm của tập lớn hơn kỷ lục **thì** lưu tập đó.
- Cuối cùng dựng `bob`, dồn phần thừa vào vùng 0.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Liệt kê bitmask `2¹²` (bản nộp chính)

**Giải thích từng đoạn:**
1. Với mỗi `mask`: `cost = Σ (a_k + 1)`, `score = Σ k` trên các bit bật.
2. Nếu `cost ≤ numArrows` và `score > best` ⇒ lưu `mask`.
3. Dựng `bob[k] = a_k + 1` cho các `k` trong mask tốt nhất; `bob[0] += numArrows − sum(bob)`.

In [9]:
def bob_bitmask(numArrows, alice):
    best_score, best_mask = -1, 0
    for mask in range(1 << 12):
        cost = score = 0
        for k in range(12):
            if mask >> k & 1:
                cost += alice[k] + 1
                score += k
        if cost <= numArrows and score > best_score:
            best_score, best_mask = score, mask
    bob = [alice[k] + 1 if best_mask >> k & 1 else 0 for k in range(12)]
    bob[0] += numArrows - sum(bob)
    return bob

for n, a, best in examples:
    got = bob_bitmask(n, a); print(got, "điểm", bob_score(a, got)); assert is_valid(n, a, got, best)

[0, 0, 0, 0, 1, 1, 0, 0, 1, 2, 3, 1] điểm 47
[0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0] điểm 27


**Độ phức tạp Cách 2:** `2¹² · 12 ≈ 5·10⁴` phép tính — hằng số, không phụ thuộc `numArrows`. Space `O(12)`.

### 7.2 Cách 3 — Quay lui (DFS) có cắt tỉa

Duyệt vùng từ 11 xuống 1, với mỗi vùng chọn "thắng" (nếu còn đủ tên) hoặc "bỏ". Cắt tỉa: nếu điểm hiện tại + tổng điểm mọi vùng còn lại `≤ best` thì dừng nhánh.

In [10]:
def bob_backtrack(numArrows, alice):
    best = [-1, None]
    suffix = [0] * 13                      # suffix[k] = k + (k-1) + ... + 1
    for k in range(1, 12):
        suffix[k] = suffix[k - 1] + k
    chosen = []
    def dfs(k, left, score):
        if score + suffix[k] <= best[0]:   # không thể vượt kỷ lục
            return
        if k == 0:
            best[0], best[1] = score, chosen[:]
            return
        need = alice[k] + 1
        if need <= left:                   # thắng vùng k
            chosen.append(k); dfs(k - 1, left - need, score + k); chosen.pop()
        dfs(k - 1, left, score)            # bỏ vùng k
    dfs(11, numArrows, 0)
    bob = [0] * 12
    for k in best[1]:
        bob[k] = alice[k] + 1
    bob[0] += numArrows - sum(bob)
    return bob

for n, a, best in examples:
    got = bob_backtrack(n, a); assert is_valid(n, a, got, best)
print([bob_backtrack(n, a) for n, a, _ in examples])

[[0, 0, 0, 0, 1, 1, 0, 0, 1, 2, 3, 1], [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0]]


**Độ phức tạp Cách 3:** worst case `O(2¹¹)` nút, thực tế ít hơn nhiều nhờ cắt tỉa.

### 7.3 Cách 4 — DP cái túi có truy vết

`dp[i][c]` = điểm tốt nhất khi chỉ xét các vùng `1..i` với `c` mũi tên. Truy vết ngược để biết vùng nào thắng. `O(12 · numArrows)` — vẫn nhanh với `10⁵`, và là cách tổng quát khi số vùng lớn nhưng tổng mũi tên nhỏ.

In [11]:
def bob_knapsack(numArrows, alice):
    W = numArrows
    dp = [[0] * (W + 1) for _ in range(12)]          # dp[i][c], i = 0..11 (vùng 0 vô nghĩa)
    for i in range(1, 12):
        w = alice[i] + 1
        prev, cur = dp[i - 1], dp[i]
        for c in range(W + 1):
            cur[c] = prev[c]
            if c >= w and prev[c - w] + i > cur[c]:
                cur[c] = prev[c - w] + i
    bob = [0] * 12; c = W
    for i in range(11, 0, -1):                       # truy vết
        if dp[i][c] != dp[i - 1][c]:
            bob[i] = alice[i] + 1; c -= bob[i]
    bob[0] += numArrows - sum(bob)
    return bob

for n, a, best in examples:
    got = bob_knapsack(n, a); assert is_valid(n, a, got, best)
print([bob_knapsack(n, a) for n, a, _ in examples])

[[0, 0, 0, 0, 1, 1, 0, 0, 1, 2, 3, 1], [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0]]


**Độ phức tạp Cách 4:** `O(12 · numArrows)` thời gian và bộ nhớ (`1,2·10⁶` ô khi `numArrows = 10⁵`).

### 7.4 Cách 5 — `itertools.combinations` theo kích thước tập

Một cách viết khác của Cách 2: với mỗi kích thước `r`, duyệt mọi tập `r` vùng. Dễ đọc, cùng độ phức tạp.

In [12]:
def bob_combinations(numArrows, alice):
    best = (-1, ())
    for r in range(12):
        for S in combinations(range(1, 12), r):
            if sum(alice[k] + 1 for k in S) <= numArrows:
                best = max(best, (sum(S), S))
    bob = [0] * 12
    for k in best[1]: bob[k] = alice[k] + 1
    bob[0] += numArrows - sum(bob)
    return bob

for n, a, best in examples:
    assert is_valid(n, a, bob_combinations(n, a), best)
print([bob_combinations(n, a) for n, a, _ in examples])

[[0, 0, 0, 0, 1, 1, 0, 0, 1, 2, 3, 1], [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0]]


## 8. TRACE từng bước

Trace Cách 3 (quay lui) trên ví dụ 2 (`numArrows = 3`): in mỗi lần tìm được kỷ lục mới và mỗi lần cắt tỉa ở tầng trên cùng.

| Bước | Nhánh | Mũi còn | Điểm | Ghi chú |
|---|---|---|---|---|
| 1 | thắng 11 (tốn 3) | 0 | 11 | không còn tên ⇒ các vùng sau bỏ ⇒ kỷ lục 11 |
| 2 | bỏ 11, thắng 10, 9, 8 | 0 | 27 | kỷ lục mới 27 |
| 3 | bỏ 11, thắng 10, 9, bỏ 8, thắng 7 | 0 | 26 | 26 ≤ 27 |
| … | các nhánh khác | | | bị cắt tỉa vì `điểm + tổng còn lại ≤ 27` |

In [13]:
def trace_backtrack(numArrows, alice):
    best = [-1]; suffix = [0] * 13
    for k in range(1, 12): suffix[k] = suffix[k - 1] + k
    chosen = []; nodes = [0]
    def dfs(k, left, score):
        nodes[0] += 1
        if score + suffix[k] <= best[0]:
            return
        if k == 0:
            best[0] = score
            print(f"kỷ lục mới {score}: thắng các vùng {chosen}, còn {left} mũi")
            return
        need = alice[k] + 1
        if need <= left:
            chosen.append(k); dfs(k - 1, left - need, score + k); chosen.pop()
        dfs(k - 1, left, score)
    dfs(11, numArrows, 0)
    print(f"tổng số nút đã thăm: {nodes[0]} (so với 2^12 = 4096 tập)")
    return best[0]

print("Đáp án:", trace_backtrack(*examples[1][:2]))
print()
print("Đáp án:", trace_backtrack(*examples[0][:2]))

kỷ lục mới 11: thắng các vùng [11], còn 0 mũi
kỷ lục mới 27: thắng các vùng [10, 9, 8], còn 0 mũi
tổng số nút đã thăm: 140 (so với 2^12 = 4096 tập)
Đáp án: 27

kỷ lục mới 45: thắng các vùng [11, 10, 9, 8, 7], còn 0 mũi
kỷ lục mới 47: thắng các vùng [11, 10, 9, 8, 5, 4], còn 0 mũi
tổng số nút đã thăm: 104 (so với 2^12 = 4096 tập)
Đáp án: 47


In [14]:
def trace_knapsack_table(numArrows, alice):
    W = numArrows; dp = [[0] * (W + 1) for _ in range(12)]
    for i in range(1, 12):
        w = alice[i] + 1
        for c in range(W + 1):
            dp[i][c] = max(dp[i - 1][c], dp[i - 1][c - w] + i if c >= w else -1)
        print(f"sau vùng {i:2d} (giá {w}): dp[c] với c=0..{W} = {dp[i]}")

trace_knapsack_table(*examples[1][:2])

sau vùng  1 (giá 1): dp[c] với c=0..3 = [0, 1, 1, 1]
sau vùng  2 (giá 2): dp[c] với c=0..3 = [0, 1, 2, 3]
sau vùng  3 (giá 1): dp[c] với c=0..3 = [0, 3, 4, 5]
sau vùng  4 (giá 1): dp[c] với c=0..3 = [0, 4, 7, 8]
sau vùng  5 (giá 1): dp[c] với c=0..3 = [0, 5, 9, 12]
sau vùng  6 (giá 1): dp[c] với c=0..3 = [0, 6, 11, 15]
sau vùng  7 (giá 1): dp[c] với c=0..3 = [0, 7, 13, 18]
sau vùng  8 (giá 1): dp[c] với c=0..3 = [0, 8, 15, 21]
sau vùng  9 (giá 1): dp[c] với c=0..3 = [0, 9, 17, 24]
sau vùng 10 (giá 1): dp[c] với c=0..3 = [0, 10, 19, 27]
sau vùng 11 (giá 3): dp[c] với c=0..3 = [0, 10, 19, 27]


## 9. Code nộp LeetCode

In [15]:
class Solution:
    def maximumBobPoints(self, numArrows: int, aliceArrows: List[int]) -> List[int]:
        best_score, best_mask = -1, 0
        for mask in range(1 << 12):                 # tập vùng Bob thắng
            cost = score = 0
            for k in range(12):
                if mask >> k & 1:
                    cost += aliceArrows[k] + 1      # cần nhiều hơn Alice đúng 1 mũi
                    score += k
            if cost <= numArrows and score > best_score:
                best_score, best_mask = score, mask
        bob = [aliceArrows[k] + 1 if best_mask >> k & 1 else 0 for k in range(12)]
        bob[0] += numArrows - sum(bob)              # mũi thừa dồn vào vùng 0 (0 điểm)
        return bob

for n, a, best in examples:
    assert is_valid(n, a, Solution().maximumBobPoints(n, a), best)
print("Ví dụ đề: OK", [Solution().maximumBobPoints(n, a) for n, a, _ in examples])

Ví dụ đề: OK [[0, 0, 0, 0, 1, 1, 0, 0, 1, 2, 3, 1], [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0]]


## 10. Vì sao thuật toán đúng

**Chuẩn hóa (Bổ đề ở mục 6).** Cho phương án tối ưu `b`. Với mỗi vùng `k ≥ 1`:
- nếu `b_k > a_k` (Bob thắng): thay `b_k` bằng `a_k + 1 ≤ b_k` — vẫn thắng;
- nếu `b_k ≤ a_k` (Bob thua hoặc không ai được): thay `b_k` bằng `0` — vẫn không được điểm.

Tổng số mũi giảm (hoặc giữ nguyên); dồn phần giảm vào vùng `0`. Điểm không đổi (vùng 0 đáng 0 điểm; nếu Bob "thắng" vùng 0 cũng chỉ được 0). Vậy tồn tại phương án tối ưu có dạng "chọn tập `S`, `b_k = a_k + 1` với `k ∈ S`, phần còn lại ở vùng 0".

**Liệt kê đầy đủ.** Cách 2 duyệt **mọi** tập `S` (2¹² mask) và chọn tập khả thi có điểm lớn nhất ⇒ đạt tối ưu. Phương án trả về có tổng đúng `numArrows` (nhờ bước dồn), các phần tử không âm (vì `cost ≤ numArrows`), và điểm = `best_score`.

**DP (Cách 4).** `dp[i][c]` thỏa truy hồi cái túi chuẩn; truy vết so `dp[i][c]` với `dp[i−1][c]` để biết món `i` có được chọn không.

In [16]:
# Kiểm chứng chuẩn hóa: với mọi cách chia (numArrows nhỏ), điểm không vượt quá điểm tốt nhất của dạng chuẩn
rng = random.Random(10)
for _ in range(30):
    N = rng.randint(1, 4)
    alice = [0] * 12
    for _ in range(N): alice[rng.randrange(12)] += 1
    best_norm = bob_score(alice, bob_bitmask(N, alice))
    assert max(bob_score(alice, b) for b in all_distributions(N, 12)) == best_norm
print("Điểm tốt nhất trên MỌI cách chia == điểm tốt nhất của dạng chuẩn (30 bộ, N <= 4)")

Điểm tốt nhất trên MỌI cách chia == điểm tốt nhất của dạng chuẩn (30 bộ, N <= 4)


## 11. Phân tích Time / Space

Gọi `N = numArrows`, `R = 12` vùng.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn mọi cách chia | `O(C(N+11, 11) · R)` | `O(R)` |
| 2. Bitmask | `O(2^R · R)` ≈ `5·10⁴` | `O(R)` |
| 3. Quay lui + cắt tỉa | `O(2^R)` worst | `O(R)` |
| 4. DP cái túi | `O(R · N)` ≈ `1,2·10⁶` | `O(R · N)` |
| 5. combinations | `O(2^R · R)` | `O(R)` |

Bitmask không phụ thuộc `N` — đây là ví dụ điển hình "số đối tượng nhỏ (12) ⇒ duyệt tập con".

In [17]:
rng = random.Random(2212)
N = 10**5
alice = [0] * 12
for _ in range(N): alice[rng.randrange(12)] += 1
alice2 = [0] * 11 + [N]                         # Alice dồn hết vào vùng 11
for name, fn in [("bitmask", bob_bitmask), ("backtrack", bob_backtrack), ("knapsack", bob_knapsack),
                 ("combinations", bob_combinations), ("Solution", lambda n, a: Solution().maximumBobPoints(n, a))]:
    t = time.perf_counter(); b1 = fn(N, alice); b2 = fn(N, alice2)
    print(f"{name:12s}: điểm {bob_score(alice, b1)}, {bob_score(alice2, b2)} ({time.perf_counter() - t:.3f}s)")

bitmask     : điểm 66, 55 (0.005s)
backtrack   : điểm 66, 55 (0.000s)


knapsack    : điểm 66, 55 (0.148s)
combinations: điểm 66, 55 (0.002s)
Solution    : điểm 66, 55 (0.004s)


## 12. Bộ test

- Ví dụ đề (bằng `is_valid`).
- Biên: `numArrows = 1`; Alice dồn hết vào vùng 11; Alice dồn hết vào vùng 0; Alice rải đều.
- Đối chiếu **điểm** của mọi cách với vét cạn mọi cách chia (`numArrows ≤ 4`), và giữa các cách với nhau ở `numArrows` lớn (seed cố định); mọi đầu ra đều phải hợp lệ.

In [18]:
fns = {"bitmask": bob_bitmask, "backtrack": bob_backtrack, "knapsack": bob_knapsack,
       "combinations": bob_combinations, "Solution": lambda n, a: Solution().maximumBobPoints(n, a)}
def check_case(n, a, best=None):
    if best is None:
        best = bob_score(a, bob_bitmask(n, a))
    for name, fn in fns.items():
        assert is_valid(n, a, fn(n, a), best), (name, n, a)
    return best

for n, a, best in examples:
    check_case(n, a, best)
assert check_case(1, [0] * 11 + [1]) == 10          # bắn 1 mũi vào vùng 10 (Alice không bắn)
assert check_case(5, [0] * 11 + [5]) == 10 + 9 + 8 + 7 + 6  # vùng 11 cần 6 mũi > 5: lấy 5 vùng giá 1
assert check_case(5, [5] + [0] * 11) == 11 + 10 + 9 + 8 + 7
print("Ví dụ + biên: OK")

Ví dụ + biên: OK


In [19]:
rng = random.Random(42)
for _ in range(60):
    N = rng.randint(1, 4); a = [0] * 12
    for _ in range(N): a[rng.randrange(12)] += 1
    best = max(bob_score(a, b) for b in all_distributions(N, 12))
    check_case(N, a, best)
for _ in range(100):
    N = rng.randint(1, 300); a = [0] * 12
    for _ in range(N): a[rng.choice([rng.randrange(12), 11, 10])] += 1
    check_case(N, a)
print("60 test nhỏ khớp vét cạn mọi cách chia; 100 test lớn: mọi cách hợp lệ và cùng điểm")

60 test nhỏ khớp vét cạn mọi cách chia; 100 test lớn: mọi cách hợp lệ và cùng điểm


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Không dồn mũi tên thừa (tổng ≠ `numArrows`)

In [20]:
def wrong_no_leftover(numArrows, alice):
    best_score, best_mask = -1, 0
    for mask in range(1 << 12):
        cost = sum(alice[k] + 1 for k in range(12) if mask >> k & 1)
        score = sum(k for k in range(12) if mask >> k & 1)
        if cost <= numArrows and score > best_score: best_score, best_mask = score, mask
    return [alice[k] + 1 if best_mask >> k & 1 else 0 for k in range(12)]   # SAI: quên dồn mũi thừa

# cần một ca mà tập tối ưu KHÔNG dùng hết tên (ví dụ đề tình cờ dùng hết): tự dựng
n, a = 13, [2] + [1] * 11            # mỗi vùng 1..11 tốn 2 mũi -> 6 vùng tốn 12, thừa 1
best = bob_score(a, bob_bitmask(n, a))
b = wrong_no_leftover(n, a)
print("sai:", b, "tổng =", sum(b), "!= numArrows =", n, "| hợp lệ?", is_valid(n, a, b, best))

sai: [0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 2] tổng = 12 != numArrows = 13 | hợp lệ? False


In [21]:
b = Solution().maximumBobPoints(n, a)
print("sửa:", b, "tổng =", sum(b), "| hợp lệ?", is_valid(n, a, b, best))

sửa: [1, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 2] tổng = 13 | hợp lệ? True


### Lỗi 2 — Bắn đúng `a_k` mũi (hòa) và tưởng là thắng

Hòa thì **Alice** được điểm. Giá đúng là `a_k + 1`.

In [22]:
def wrong_tie(numArrows, alice):
    best_score, best_mask = -1, 0
    for mask in range(1 << 12):
        cost = sum(alice[k] for k in range(12) if mask >> k & 1)          # SAI: thiếu +1
        score = sum(k for k in range(12) if mask >> k & 1)
        if cost <= numArrows and score > best_score: best_score, best_mask = score, mask
    bob = [alice[k] if best_mask >> k & 1 else 0 for k in range(12)]
    bob[0] += numArrows - sum(bob)
    return bob

n, a, best = examples[1]
b = wrong_tie(n, a)
print("sai:", b, "điểm thật =", bob_score(a, b), "| tối ưu:", best)

sai: [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 2] điểm thật = 0 | tối ưu: 27


In [23]:
print("sửa:", Solution().maximumBobPoints(n, a), "điểm", bob_score(a, Solution().maximumBobPoints(n, a)))

sửa: [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0] điểm 27


### Lỗi 3 — Tham lam "vùng điểm cao nhất trước"

In [24]:
def wrong_greedy_high(numArrows, alice):
    bob = [0] * 12; left = numArrows
    for k in range(11, 0, -1):
        if alice[k] + 1 <= left:
            bob[k] = alice[k] + 1; left -= bob[k]
    bob[0] += left
    return bob

n, a, best = examples[1]
print("sai: điểm", bob_score(a, wrong_greedy_high(n, a)), "| tối ưu:", best)

sai: điểm 11 | tối ưu: 27


In [25]:
print("sửa (duyệt hết tập con):", bob_score(a, Solution().maximumBobPoints(n, a)))

sửa (duyệt hết tập con): 27


### Lỗi 4 — Tham lam theo tỉ lệ "điểm / giá"

Trông hợp lý (giống cái túi phân số) nhưng sai với cái túi 0/1. Để máy tìm phản ví dụ:

In [26]:
def wrong_greedy_ratio(numArrows, alice):
    order = sorted(range(1, 12), key=lambda k: k / (alice[k] + 1), reverse=True)
    bob = [0] * 12; left = numArrows
    for k in order:
        if alice[k] + 1 <= left:
            bob[k] = alice[k] + 1; left -= bob[k]
    bob[0] += left
    return bob

rng = random.Random(7)
for _ in range(10000):
    N = rng.randint(2, 30); a = [0] * 12
    for _ in range(N): a[rng.randrange(12)] += 1
    if bob_score(a, wrong_greedy_ratio(N, a)) < bob_score(a, bob_bitmask(N, a)):
        break
print("phản ví dụ: numArrows =", N, "alice =", a)
print("sai: điểm", bob_score(a, wrong_greedy_ratio(N, a)), "| tối ưu:", bob_score(a, bob_bitmask(N, a)))

phản ví dụ: numArrows = 22 alice = [3, 2, 3, 1, 2, 0, 2, 0, 3, 4, 2, 0]
sai: điểm 59 | tối ưu: 60


In [27]:
print("sửa:", bob_score(a, Solution().maximumBobPoints(N, a)))

sửa: 60


### Lỗi 5 — DP cái túi duyệt sức chứa **tăng dần** trên mảng 1 chiều

Duyệt `c` tăng dần cho phép dùng một vùng **nhiều lần** (thành cái túi không giới hạn) ⇒ điểm ảo cao hơn thực tế.

In [28]:
def wrong_unbounded_value(numArrows, alice):
    dp = [0] * (numArrows + 1)
    for k in range(1, 12):
        w = alice[k] + 1
        for c in range(w, numArrows + 1):                  # SAI: tăng dần
            dp[c] = max(dp[c], dp[c - w] + k)
    return dp[numArrows]

n, a, best = examples[1]
print("sai (giá trị DP):", wrong_unbounded_value(n, a), "| đúng:", best)

sai (giá trị DP): 30 | đúng: 27


In [29]:
print("sửa (giảm dần):", knapsack_value([a[k] + 1 for k in range(1, 12)], list(range(1, 12)), n))

sửa (giảm dần): 27


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Số đối tượng rất nhỏ (≤ 20)** ⇒ nghĩ ngay tới **duyệt tập con bằng bitmask**, kể cả khi các số khác (như `numArrows`) rất lớn.
- **"Thắng/thua từng phần, giá để thắng cố định"** ⇒ cái túi 0/1; trước tiên **chuẩn hóa** để mỗi phần chỉ có 2 lựa chọn.
- Đầu ra có ràng buộc tổng ⇒ nhớ **dồn phần dư** vào chỗ vô hại.
- Bài nhiều đáp án ⇒ viết `is_valid` và so **điểm** thay vì so mảng.
- Tham lam (giá trị cao / tỉ lệ cao) cho cái túi 0/1 thường sai — luôn tìm phản ví dụ bằng máy.

## 15. Biến thể của bài

1. **Tối đa chênh lệch điểm** (Bob − Alice) (bài tập 5): mỗi vùng Bob thắng còn lấy mất điểm của Alice nếu Alice đã bắn vào đó ⇒ giá trị món `k` là `2k` nếu `a_k > 0`, `k` nếu `a_k = 0`.
2. **Số vùng lớn (ví dụ 40)** ⇒ `2⁴⁰` quá lớn; dùng DP cái túi `O(R · N)` hoặc "meet in the middle".
3. **LeetCode 1601, 698, 2305** — các bài duyệt tập con / bitmask với số đối tượng nhỏ.
4. **Bob muốn tối thiểu số mũi tên để đạt ít nhất `P` điểm:** cái túi "ngược" (giá trị ↔ trọng lượng).

Minh họa biến thể 4 (tự dựng, kiểm bằng bitmask):

In [30]:
def min_arrows_for_points(alice, P):
    INF = float("inf")
    dp = [0] + [INF] * P                     # dp[p] = số mũi ít nhất để đạt >= p điểm (cắt ở P)
    for k in range(1, 12):
        w = alice[k] + 1
        for p in range(P, -1, -1):
            np_ = min(P, p + k)
            if dp[p] + w < dp[np_]:
                dp[np_] = dp[p] + w
    return dp[P] if dp[P] < INF else -1

def min_arrows_brute(alice, P):
    best = -1
    for mask in range(1 << 12):
        sc = sum(k for k in range(12) if mask >> k & 1)
        if sc >= P:
            c = sum(alice[k] + 1 for k in range(12) if mask >> k & 1)
            best = c if best == -1 or c < best else best
    return best

rng = random.Random(4)
for _ in range(200):
    a = [rng.randint(0, 4) for _ in range(12)]; P = rng.randint(0, 70)
    assert min_arrows_for_points(a, P) == min_arrows_brute(a, P)
print("Biến thể 'ít mũi nhất để đạt P điểm' khớp bitmask; ví dụ 1, P=47 ->", min_arrows_for_points(examples[0][1], 47))

Biến thể 'ít mũi nhất để đạt P điểm' khớp bitmask; ví dụ 1, P=47 -> 9


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn chia mũi | mọi phân phối | `O(C(N+11,11))` | `O(R)` | chuẩn kiểm tra, `N ≤ 5` |
| 2. Bitmask | mọi tập vùng thắng | `O(2^R·R)` | `O(R)` | **nộp bài** |
| 3. Quay lui + cắt tỉa | DFS từ vùng 11 | `O(2^R)` | `O(R)` | khi muốn cắt tỉa |
| 4. DP cái túi | `dp[i][c]` + truy vết | `O(R·N)` | `O(R·N)` | `R` lớn, `N` vừa |
| 5. combinations | tập con theo kích thước | `O(2^R·R)` | `O(R)` | dễ đọc |

## 17. Bài tập (từ dễ đến khó)

1. **Đoán output (điểm tối ưu):** `numArrows = 1`, Alice bắn 1 mũi vào vùng 0.
2. **Tính tay:** `numArrows = 4`, `alice = [0,0,0,0,0,0,0,0,0,1,1,2]` — liệt kê vài tập vùng và chọn tốt nhất.
3. **Điền TODO:** `score_and_cost(mask, alice)` trả về `(điểm, giá)` của một tập.
4. **Tự viết lại:** viết lại Cách 2 không nhìn code.
5. **Mini challenge:** `max_margin(numArrows, alice)` — tối đa `(điểm Bob) − (điểm Alice)`. Kiểm bằng vét cạn mọi cách chia với `numArrows ≤ 4`.

In [31]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def score_and_cost(mask, alice):
    # TODO
    pass

In [32]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
def my_maximum_bob_points(numArrows, alice):
    # TODO
    pass

In [33]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
def max_margin(numArrows, alice):
    # TODO: giá trị món k là 2k nếu alice[k] > 0, ngược lại k; điểm Alice ban đầu = tổng k có alice[k] > 0
    pass

## 18. Gợi ý 3 mức

**Bài 1.** (1) Bob có 1 mũi. (2) Vùng 11 Alice không bắn ⇒ giá 1. (3) Điểm tối ưu 11.

**Bài 2.** (1) Giá: vùng 11 là 3, vùng 10 là 2, vùng 9 là 2, vùng 8..1 là 1. (2) Ứng viên: `{11, 8}` (giá 4, điểm 19), `{10, 9}` (giá 4, điểm 19), `{10, 8, 7}` (giá 4, điểm 25), `{8, 7, 6, 5}` (giá 4, điểm 26). (3) Tối ưu 26.

**Bài 3.** (1) Duyệt `k = 0..11`. (2) Bit bật ⇒ cộng `k` vào điểm, `a_k + 1` vào giá. (3) Trả tuple.

**Bài 4.** (1) `for mask in range(4096)`. (2) Giữ mask tốt nhất. (3) Dồn phần thừa vào `bob[0]`.

**Bài 5.** (1) Điểm Alice = `Σ_{k: a_k > 0} k` trừ đi phần Bob lấy được. (2) Bob thắng vùng `k` có `a_k > 0` ⇒ chênh lệch tăng `2k`; vùng `a_k = 0` ⇒ tăng `k`. (3) Bitmask với giá trị mới.

## 19. Đáp án tham khảo

In [34]:
# Bài 1, 2
assert bob_score([1] + [0] * 11, Solution().maximumBobPoints(1, [1] + [0] * 11)) == 11
a2 = [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 2]
assert bob_score(a2, Solution().maximumBobPoints(4, a2)) == 26
print("Bài 1: 11 | Bài 2: 26 với", Solution().maximumBobPoints(4, a2))

Bài 1: 11 | Bài 2: 26 với [0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0]


In [35]:
# Bài 3, 4
def score_and_cost_ans(mask, alice):
    sc = sum(k for k in range(12) if mask >> k & 1)
    cost = sum(alice[k] + 1 for k in range(12) if mask >> k & 1)
    return sc, cost

my_maximum_bob_points_ans = bob_bitmask
assert score_and_cost_ans(0b111 << 8, a2) == (27, 5)
rng = random.Random(3)
for _ in range(100):
    N = rng.randint(1, 50); a = [0] * 12
    for _ in range(N): a[rng.randrange(12)] += 1
    best = bob_score(a, bob_knapsack(N, a))
    assert is_valid(N, a, my_maximum_bob_points_ans(N, a), best)
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [36]:
# Bài 5
def alice_score(alice, bob):
    return sum(k for k in range(12) if alice[k] >= bob[k] and not (alice[k] == 0 and bob[k] == 0))

def max_margin_ans(numArrows, alice):
    base_alice = sum(k for k in range(12) if alice[k] > 0)
    best = None
    for mask in range(1 << 12):
        cost = sum(alice[k] + 1 for k in range(12) if mask >> k & 1)
        if cost > numArrows: continue
        gain = sum((2 * k if alice[k] > 0 else k) for k in range(12) if mask >> k & 1)
        val = gain - base_alice
        best = val if best is None or val > best else best
    return best

rng = random.Random(5)
for _ in range(40):
    N = rng.randint(1, 4); a = [0] * 12
    for _ in range(N): a[rng.randrange(12)] += 1
    brute = max(bob_score(a, b) - alice_score(a, b) for b in all_distributions(N, 12))
    assert max_margin_ans(N, a) == brute
print("Bài 5: OK; ví dụ 1 ->", max_margin_ans(*examples[0][:2]))

Bài 5: OK; ví dụ 1 -> 35


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "duyệt tập con nhỏ":**

```python
best, best_mask = -1, 0
for mask in range(1 << R):
    cost = sum(w[k] for k in range(R) if mask >> k & 1)
    val  = sum(v[k] for k in range(R) if mask >> k & 1)
    if cost <= CAP and val > best:
        best, best_mask = val, mask
# dựng đáp án từ best_mask, xử lý phần dư
```

In [37]:
def template(numArrows, alice):
    best, best_mask = -1, 0
    for mask in range(1 << 12):
        cost = sum(alice[k] + 1 for k in range(12) if mask >> k & 1)
        val = sum(k for k in range(12) if mask >> k & 1)
        if cost <= numArrows and val > best: best, best_mask = val, mask
    bob = [alice[k] + 1 if best_mask >> k & 1 else 0 for k in range(12)]
    bob[0] += numArrows - sum(bob)
    return bob
print([template(n, a) for n, a, _ in examples])

[[0, 0, 0, 0, 1, 1, 0, 0, 1, 2, 3, 1], [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0]]


**Checklist:**

- [ ] Giá thắng vùng `k` là `a_k + 1` (hòa thuộc về Alice).
- [ ] Mỗi vùng chỉ 2 lựa chọn: `0` hoặc `a_k + 1`.
- [ ] Duyệt đủ `2¹²` tập (không tham lam).
- [ ] Dồn mũi thừa để tổng **đúng** `numArrows`.
- [ ] Nếu dùng DP 1 chiều: duyệt sức chứa **giảm dần**.
- [ ] Test bằng `is_valid` (so điểm), không so mảng cụ thể.